In [215]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
# from xgboost import XGBClassifier
# from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler,OrdinalEncoder
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report

In [216]:
df = pd.read_csv('telco_data.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1.0,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34.0,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2.0,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45.0,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2.0,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [217]:
df.drop('customerID',axis=1,inplace=True)

In [218]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            6293 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           6043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            4543 non-null   float64
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   6043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       5543 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [219]:
X = df.drop('Churn',axis=1)
y = df.Churn

In [220]:
num_cols = X.select_dtypes(include='number').columns
obj_cols = X.select_dtypes(include='object').columns

In [221]:
xtrain,xtest,ytrain,ytest = train_test_split(X,y,random_state=42,train_size=0.8)

In [222]:
num_pipeline = Pipeline(
    steps=[
        ('num_imputer',SimpleImputer(strategy='median')),
        ('scaling',StandardScaler())
    ]
)
cat_pipeline = Pipeline(
    steps=[
        ('cat_imputer',SimpleImputer(strategy='constant',fill_value='Unknown')),
        ('encoder',OrdinalEncoder())
    ]
)

In [223]:
preprocessing = ColumnTransformer(transformers=[
    ('num_preprocessing',num_pipeline,num_cols),
    (('cat_preprocessing',cat_pipeline,obj_cols))
])

X = preprocessing.fit_transform(X)

In [224]:
model = KMeans(n_clusters=5)
model.fit(X)

,n_clusters,5
,init,'k-means++'
,n_init,'auto'
,max_iter,300
,tol,0.0001
,verbose,0
,random_state,None
,copy_x,True
,algorithm,'lloyd'


In [225]:
model.fit(X)
model.predict(X)

array([0, 4, 2, ..., 0, 0, 1], shape=(7043,), dtype=int32)

In [226]:
X = model.transform(X)
X

array([[ 639.81610461, 3332.30622198, 1902.25764737, 1951.83401574,
         676.10245667],
       [1678.80814555, 4371.30532519,  863.26129881, 2990.83181748,
         362.9247519 ],
       [2987.8067721 , 5680.30517979,  445.76214547, 4299.83147163,
        1671.91160249],
       ...,
       [ 150.85416184, 2843.30662945, 2391.25680239, 1462.83540596,
        1165.09811876],
       [ 484.82108324, 3177.30628424, 2057.25802499, 1796.83443943,
         831.10175907],
       [2262.20027819,  430.32482872, 4804.25656267,  950.17957917,
        3578.09576039]], shape=(7043, 5))

In [227]:
log_model = LogisticRegression()
log_model.fit(X,y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [228]:
log_model.predict(X)

array(['No', 'No', 'No', ..., 'No', 'No', 'No'],
      shape=(7043,), dtype=object)

In [229]:
y

0        No
1        No
2       Yes
3        No
4       Yes
       ... 
7038     No
7039     No
7040     No
7041    Yes
7042     No
Name: Churn, Length: 7043, dtype: object

In [230]:
log_model.score(X,y)

0.7346301292063041

In [231]:
preprocessing = ColumnTransformer(transformers=[
    ('num_preprocessing',num_pipeline,num_cols),
    (('cat_preprocessing',cat_pipeline,obj_cols))
])

# X = preprocessing.fit_transform(X)

# main_pipeline = Pipeline(
#     steps=[
#         ('preprocessing',preprocessing),
#         ('kmeans_',KMeans(n_clusters=2)),
#         ('model_',LogisticRegression(class_weight='balanced'))
#     ]
# )
main_pipeline = Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('kmeans_',KMeans(n_clusters=2)),
        ('model_',KNeighborsClassifier())
    ]
)
# main_pipeline = Pipeline(
#     steps=[
#         ('preprocessing',preprocessing),
#         ('kmeans_',KMeans(n_clusters=2)),
#         ('model_',GaussianNB())
#     ]
# )

In [232]:
ytrain = ytrain.map({'Yes':1,'No':0})

In [233]:
main_pipeline.fit(xtrain,ytrain)

,steps,"[('preprocessing', ...), ('kmeans_', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num_preprocessing', ...), ('cat_preprocessing', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [234]:
main_pipeline.score(xtrain,ytrain)

0.7847000354987576

In [235]:
ytrain_pred = main_pipeline.predict(xtrain)


In [236]:
print(classification_report(ytrain,ytrain_pred))

              precision    recall  f1-score   support

           0       0.80      0.94      0.87      4138
           1       0.68      0.35      0.47      1496

    accuracy                           0.78      5634
   macro avg       0.74      0.65      0.67      5634
weighted avg       0.77      0.78      0.76      5634



In [237]:
import pickle

with open('model_pipe.pkl','wb') as file:
    pickle.dump(main_pipeline,file)